# Food Delivery Uplift Modeling

Promo-code targeting with S-, T-, and X-learner uplift models, hyperparameter tuning, and an inference wrapper.

Outputs were stripped for public release. Re-run the notebook after configuring the required private data access or local artifact paths.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklift.metrics import uplift_auc_score, qini_auc_score, uplift_at_k
from causalml.inference.meta import BaseSClassifier,BaseXClassifier
from sklift.metrics import uplift_at_k
from optuna import create_study



pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


In [ ]:
# See the project README for context.
DATA_PATH = 'uplift_fp_data.csv'
df = pd.read_csv(DATA_PATH)
df_original = df.copy()


In [ ]:
print('shape:', df.shape)
display(df.head(10))
print(df.info())


In [ ]:
# See the project README for context.
na_counts = df.isna().sum()
print("Step completed")


In [ ]:
# See the project README for context.
for col in ['history_segment', 'zip_code', 'channel', 'treatment', 'target', 'mens', 'womens', 'newbie']:
    if col in df.columns:
        vals = df[col].unique()
        print(col, np.sort(vals)[:10])


In [ ]:
conv = (df
    .groupby('treatment')['target']
    .agg(['mean','sum','count'])
    .rename(columns={'mean':'conversion_rate','sum':'conversions','count':'n'})
    .reset_index()
)
conv


In [ ]:
# See the project README for context.
ct = pd.crosstab(df['treatment'], df['target'])
ct


In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='treatment')
# Plot title removed during public cleanup.
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='treatment', hue='target')
# Plot title removed during public cleanup.
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5,4))
sns.barplot(data=conv, x='treatment', y='conversion_rate')
# Plot title removed during public cleanup.
plt.tight_layout()
plt.show()


In [ ]:
# See the project README for context.
df[['recency','history']].describe().T


In [ ]:
# See the project README for context.
cat_cols = ['history_segment','zip_code','newbie','mens','womens','channel']
for c in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=c, hue='treatment')
    # Plot title removed during public cleanup.
    plt.tight_layout()
    plt.show()

# See the project README for context.
num_cols = ['recency','history']
for c in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(data=df, x=c, hue='treatment', kde=True, element='step', stat='density')
    # Plot title removed during public cleanup.
    plt.tight_layout()
    plt.show()

# See the project README for context.
for c in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x='target', y=c)
    # Plot title removed during public cleanup.
    plt.tight_layout()
    plt.show()


In [ ]:
def segment_table(col):
    g = (df.groupby([col,'treatment'])['target']
           .mean()
           .unstack()
           .rename(columns={0:'conv_ctrl',1:'conv_treat'}))
    g['uplift_pp'] = (g['conv_treat'] - g['conv_ctrl']) * 100
    return g.sort_values('uplift_pp', ascending=False)

# See the project README for context.
for c in ['history_segment','zip_code','newbie','mens','womens','channel']:
    tmp = segment_table(c)
    display(tmp.round(4))
    plt.figure(figsize=(7,4))
    sns.barplot(data=tmp.reset_index(), x=c, y='uplift_pp')
    plt.title(f"Segment uplift for {c}, pp")
    plt.axhline(0, color='gray', linewidth=1)
    plt.tight_layout()
    plt.show()


In [ ]:
# See the project README for context.
n_treat = (df.treatment == 1).sum()
n_ctrl  = (df.treatment == 0).sum()
s_treat = df.loc[df.treatment == 1, 'target'].sum()
s_ctrl  = df.loc[df.treatment == 0, 'target'].sum()

p_treat = s_treat / n_treat
p_ctrl  = s_ctrl  / n_ctrl
diff = p_treat - p_ctrl

# See the project README for context.
z_stat, p_val = proportions_ztest([s_treat, s_ctrl], [n_treat, n_ctrl], alternative='two-sided')


In [ ]:
# See the project README for context.
ci_treat = proportion_confint(s_treat, n_treat, alpha=0.05, method='wilson')
ci_ctrl  = proportion_confint(s_ctrl,  n_ctrl,  alpha=0.05, method='wilson')
se_diff = np.sqrt(p_treat*(1-p_treat)/n_treat + p_ctrl*(1-p_ctrl)/n_ctrl)
ci_diff = (diff - 1.96*se_diff, diff + 1.96*se_diff)

print(f'n_treat={n_treat}, successes={s_treat}, rate={p_treat:.4f}, 95% CI={ci_treat}')
print(f'n_ctrl ={n_ctrl }, successes={s_ctrl }, rate={p_ctrl :.4f}, 95% CI={ci_ctrl }')
print(f'diff={diff:.4f}, 95% CI for diff={ci_diff}')
print(f'z={z_stat:.3f}, p-value={p_val:.4g}')

# See the project README for context.
table = np.array([[s_ctrl, n_ctrl - s_ctrl],
                  [s_treat, n_treat - s_treat]])
chi2, chi2_p, dof, _ = chi2_contingency(table, correction=False)
cramers_v = np.sqrt(chi2 / df.shape[0])
print(f'chi2={chi2:.3f}, p-value={chi2_p:.4g}, dof={dof}, CramersV={cramers_v:.3f}')


In [ ]:
corr = df.drop(columns=['treatment']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
# Plot title removed during public cleanup.
plt.tight_layout()
plt.show()

target_corr = corr['target'].drop('target').sort_values(ascending=False)

plt.figure(figsize=(6, 4))
sns.barplot(x=target_corr.values, y=target_corr.index)
# Plot title removed during public cleanup.
plt.xlabel('corr')
plt.tight_layout()
plt.show()

print(target_corr.round(3))


In [ ]:
X = df.drop(columns=['target'])
y = df['target']

# See the project README for context.
strata = df['treatment'].astype(str) + '_' + df['target'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

# See the project README for context.
treatment_train = X_train['treatment']
treatment_test = X_test['treatment']
X_train_wo_t = X_train.drop(columns=['treatment'])
X_test_wo_t = X_test.drop(columns=['treatment'])

print('train', X_train_wo_t.shape, 'test', X_test_wo_t.shape)


In [ ]:
rf_params = dict(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)

mask_t = treatment_train == 1
mask_c = treatment_train == 0

# See the project README for context.
t_model = RandomForestClassifier(**rf_params)
c_model = RandomForestClassifier(**rf_params)

# See the project README for context.
t_model.fit(X_train_wo_t[mask_t], y_train[mask_t])
c_model.fit(X_train_wo_t[mask_c], y_train[mask_c])

# See the project README for context.
proba_t = t_model.predict_proba(X_test_wo_t)[:, 1]
proba_c = c_model.predict_proba(X_test_wo_t)[:, 1]
uplift_t = proba_t - proba_c


In [ ]:
auuc = uplift_auc_score(y_test.values, uplift_t, treatment_test.values)
qauuc = qini_auc_score(y_test.values, uplift_t, treatment_test.values)
u30 = uplift_at_k(y_test.values, uplift_t, treatment_test.values, k=0.3, strategy='overall')

print(f'Uplift AUC: {auuc:.4f}')
print(f'Qini AUC:   {qauuc:.4f}')
print(f'Uplift@30:  {u30:.4f}')


In [ ]:
fi_t = pd.Series(t_model.feature_importances_, index=X_train_wo_t.columns, name='treat')
fi_c = pd.Series(c_model.feature_importances_, index=X_train_wo_t.columns, name='control')
fi = pd.concat([fi_t, fi_c], axis=1).fillna(0)
fi['mean_importance'] = fi.mean(axis=1)
fi = fi.sort_values('mean_importance', ascending=False)
display(fi.round(4))

plt.figure(figsize=(7, 4))
sns.barplot(x=fi['mean_importance'].values, y=fi.index)
# Plot title removed during public cleanup.
plt.xlabel('importance')
plt.tight_layout()
plt.show()


In [ ]:
# See the project README for context.
s_learner = BaseSClassifier(
    learner=RandomForestClassifier(**rf_params),
    control_name=0
)
s_learner.fit(
    X_train_wo_t.values,
    treatment=treatment_train.values,
    y=y_train.values
)


In [ ]:
# See the project README for context.
uplift_s = s_learner.predict(X_test_wo_t.values).squeeze()

auuc_s = uplift_auc_score(y_test.values, uplift_s, treatment_test.values)
qauuc_s = qini_auc_score(y_test.values, uplift_s, treatment_test.values)
u30_s = uplift_at_k(y_test.values, uplift_s, treatment_test.values, k=0.3, strategy='overall')

print(f'S-learner  Uplift AUC: {auuc_s:.4f}')
print(f'S-learner  Qini AUC:   {qauuc_s:.4f}')
print(f'S-learner  Uplift@30:  {u30_s:.4f}')


In [ ]:
prop_model = RandomForestClassifier(**rf_params)
prop_model.fit(X_train_wo_t, treatment_train)
p_train = prop_model.predict_proba(X_train_wo_t)[:, 1]
p_test = prop_model.predict_proba(X_test_wo_t)[:, 1]


x_learner = BaseXClassifier(
    outcome_learner=RandomForestClassifier(**rf_params),
    effect_learner=RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    control_name=0
)

x_learner.fit(
    X_train_wo_t.values,
    treatment=treatment_train.values,
    y=y_train.values,
    p=p_train
)


In [ ]:
uplift_x = x_learner.predict(X_test_wo_t.values, p=p_test).squeeze()

auuc_x = uplift_auc_score(y_test.values, uplift_x, treatment_test.values)
qauuc_x = qini_auc_score(y_test.values, uplift_x, treatment_test.values)
u30_x = uplift_at_k(y_test.values, uplift_x, treatment_test.values, k=0.3, strategy='overall')

print(f'X-learner  Uplift AUC: {auuc_x:.4f}')
print(f'X-learner  Qini AUC:   {qauuc_x:.4f}')
print(f'X-learner  Uplift@30:  {u30_x:.4f}')


In [ ]:
results = pd.DataFrame({
    'model': ['T-learner RF','S-learner RF','X-learner RF'],
    'Uplift AUC': [auuc, auuc_s, auuc_x],
    'Qini AUC':   [qauuc, qauuc_s, qauuc_x],
    'Uplift@30':  [u30, u30_s, u30_x]
})
display(results.round(4).sort_values('Uplift@30', ascending=False))


In [ ]:
# See the project README for context.
X_train_fe = X_train_wo_t.copy()
X_test_fe = X_test_wo_t.copy()

for X_ in (X_train_fe, X_test_fe):
    X_['log_history'] = np.log1p(X_['history'])  #  
    X_['both_mw'] = ((X_['mens'] == 1) & (X_['womens'] == 1)).astype(int)  #  

print('train_fe:', X_train_fe.shape, 'test_fe:', X_test_fe.shape)
display(X_train_fe.head())


In [ ]:
def cv_uplift_at30(params, X, y, t, k=0.3, splits=3):
    strata = t.astype(str) + '_' + y.astype(str)
    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, strata):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        t_tr, t_val = t.iloc[tr_idx], t.iloc[val_idx]
        m1 = t_tr == 1
        m0 = t_tr == 0
        tm = RandomForestClassifier(**params)
        cm = RandomForestClassifier(**params)
        tm.fit(X_tr[m1], y_tr[m1])
        cm.fit(X_tr[m0], y_tr[m0])
        uplift_val = tm.predict_proba(X_val)[:, 1] - cm.predict_proba(X_val)[:, 1]
        scores.append(uplift_at_k(y_val.values, uplift_val, t_val.values, k=k, strategy='overall'))
    return float(np.mean(scores))

def objective_cv(trial):
    drop_histseg = trial.suggest_categorical('drop_history_segment', [False, True])
    cols = X_train_wo_t.columns.drop('history_segment') if drop_histseg else X_train_wo_t.columns

    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 600, step=100),
        max_depth=trial.suggest_int('max_depth', 6, 12),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.8]),
        class_weight=trial.suggest_categorical('class_weight', [None, 'balanced']),
        random_state=42,
        n_jobs=-1,
    )
    return cv_uplift_at30(params, X_train_wo_t[cols], y_train, treatment_train, k=0.3, splits=3)


# Create and run the optimization study
study = create_study(direction='maximize')
study.optimize(objective_cv, n_trials=50)

# See the project README for context.
print("Best hyperparameters: ", study.best_params)
print("Best score: ", study.best_value)


In [ ]:
best = study.best_params.copy()
drop_histseg = best.pop('drop_history_segment')
best.update({'random_state':42, 'n_jobs':-1})
use_cols = X_train_wo_t.columns.drop('history_segment') if drop_histseg else X_train_wo_t.columns

tm_best = RandomForestClassifier(**best)
cm_best = RandomForestClassifier(**best)

m1 = treatment_train == 1
m0 = treatment_train == 0
tm_best.fit(X_train_wo_t[use_cols][m1], y_train[m1])
cm_best.fit(X_train_wo_t[use_cols][m0], y_train[m0])

uplift_cv_best = tm_best.predict_proba(X_test_wo_t[use_cols])[:,1] - cm_best.predict_proba(X_test_wo_t[use_cols])[:,1]

print('Uplift AUC', round(uplift_auc_score(y_test.values, uplift_cv_best, treatment_test.values), 4))
print('Qini AUC', round(qini_auc_score(y_test.values, uplift_cv_best, treatment_test.values), 4))
print('Uplift@30', round(uplift_at_k(y_test.values, uplift_cv_best, treatment_test.values, k=0.3, strategy='overall'), 4))


In [ ]:
from utils import custom_uplift_by_percentile
from sklift.viz import plot_uplift_curve, plot_qini_curve


In [ ]:
uplift_best = uplift_cv_best
fig, ax = plt.subplots(1,1,figsize=(8,6))
plot_uplift_curve(y_test.values, uplift_best.squeeze(), treatment_test.values, perfect=True, ax=ax, name='Uplift Curve')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
plot_qini_curve(y_test.values, uplift_best.squeeze(), treatment_test.values, perfect=True, ax=ax, name='Qini Curve')
plt.tight_layout(); plt.show()

_ = custom_uplift_by_percentile(
    y_true=y_test.values,
    uplift=uplift_best.squeeze(),
    treatment=treatment_test.values,
    kind='bar',
    bins=10,
    title="Uplift by percentile (improved model)"
)
plt.show()


In [ ]:
from sklift.viz import plot_uplift_curve, plot_qini_curve
from utils import custom_uplift_by_percentile

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_uplift_curve(y_test.values, uplift_t.squeeze(), treatment_test.values, perfect=True, ax=ax, name='Uplift Curve T-learner')
ax.grid(True); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_qini_curve(y_test.values, uplift_t.squeeze(), treatment_test.values, perfect=True, ax=ax, name='Qini Curve T-learner')
ax.grid(True); plt.tight_layout(); plt.show()

_ = custom_uplift_by_percentile(
    y_true=y_test.values,
    uplift=uplift_t.squeeze(),
    treatment=treatment_test.values,
    kind='bar',
    bins=10,
    title="Uplift by percentile T-learner"
)
plt.show()


In [ ]:
class UpliftModelInference:
    """Inference wrapper for a two-model uplift estimator."""

    def __init__(self, model, feature_names, logger=None):
        """Store trained treatment/control models and required features."""
        self.model = model
        self.feature_names = feature_names
        self.logger = logger
        if self.logger:
            self.logger.info(
                "UpliftModelInference initialized with features: %s",
                feature_names,
            )

    def _transform_data(self, X):
        """Return the final feature matrix expected by the model."""
        if self.logger:
            self.logger.debug("Transforming input data with shape %s", X.shape)
        return X

    def predict(self, X):
        """Return uplift scores for a pandas DataFrame of features."""
        if X.empty:
            if self.logger:
                self.logger.error("Empty input DataFrame")
            return None
        if X.isnull().any().any():
            if self.logger:
                self.logger.error("Input data contains missing values")
            return [None] * len(X)

        missing_features = set(self.feature_names) - set(X.columns)
        if missing_features:
            error_msg = f"Missing features: {missing_features}"
            if self.logger:
                self.logger.error(error_msg)
            return [None] * len(X)

        X_prepared = self._transform_data(X[self.feature_names])
        treat_pred = self.model["treat"].predict_proba(X_prepared)[:, 1]
        control_pred = self.model["control"].predict_proba(X_prepared)[:, 1]
        return treat_pred - control_pred


In [ ]:
# Use the tuned baseline models
final_models = {'treat': tm_best, 'control': cm_best}
final_features = list(use_cols)

model = UpliftModelInference(model=final_models, feature_names=final_features)


In [ ]:
test_data = pd.DataFrame({
            'recency': [1, 2, 3],
            'history_segment': [1, 2, 3], 
            'history': [100, 200, 300],
            'mens': [1, 0, 1],
            'womens': [0, 1, 0],
            'zip_code': [1, 0, 1],
            'newbie': [0, 1, 0],
            'channel': [1, 2, 0]
        })


In [ ]:

pred_demo = model.predict(test_data)
print("Example uplift predictions\n", pred_demo)


In [ ]:
pred_check = model.predict(X_test_wo_t[final_features])
mae = np.mean(np.abs(pred_check - uplift_cv_best))
print("Mean absolute difference against the reference uplift", round(mae, 8))


In [ ]:
# Score the full population
all_scores = model.predict(df[final_features])

# Score table
res = (pd.DataFrame({'user_id': df.index, 'uplift_score': all_scores})
       .sort_values('uplift_score', ascending=False))

# Top 30 percent
k = 0.30
top30 = res.head(int(len(res) * k))
top30.to_csv('target_top30.csv', index=False)
print("Saved users", len(top30))
top30.head()


In [ ]:
# Expected top-30 percent lift on the test set
u30_final = uplift_at_k(y_test.values, uplift_cv_best, treatment_test.values, k=0.3, strategy='overall')
expected_gain = u30_final * len(y_test) * k
print("Expected additional conversions for top30", round(expected_gain))


In [ ]:
# Save the model artifact
import joblib
artifact = {
    'treat_model': tm_best,
    'control_model': cm_best,
    'feature_names': final_features
}
joblib.dump(artifact, 'uplift_t_learner_rf_FINAL.joblib')
print("Model saved to uplift_t_learner_rf_FINAL.joblib")

# See the project README for context.
loaded = joblib.load('uplift_t_learner_rf_FINAL.joblib')
uplift_inf_loaded = UpliftModelInference(
    model={'treat': loaded['treat_model'], 'control': loaded['control_model']},
    feature_names=loaded['feature_names']
)
check = uplift_inf_loaded.predict(X_test_wo_t[loaded['feature_names']])
print("Matches final uplift", float(np.mean(np.abs(check - uplift_cv_best)) < 1e-12))
